# #01 **From `Dense` to `MultiDense`**

----

## 1. Imports

In [ ]:
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense

In [ ]:
from multi_dense import MultiDense

----

## 2. Ensure experimental reproductibility

In [ ]:
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

----

## 3. Define Models

In [ ]:
models: dict[str, Sequential] = dict()

### 3.1. Models

#### 3.1.1. M: **100+0**
100% ReLU + 0% Sigmoid neuron activations per hidden layer.

In [ ]:
models["100+0"] = Sequential(
    [
        MultiDense([64, 0], activations=["relu", "tanh"]),
        MultiDense([64, 0], activations=["relu", "tanh"]),
        MultiDense([32, 0], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

#### 3.1.2. M: **75+25**
75% ReLU + 35% Sigmoid neuron activations per hidden layer.<>

In [ ]:
models["75+25"] = Sequential(
    [
        MultiDense([48, 16], activations=["relu", "tanh"]),
        MultiDense([48, 16], activations=["relu", "tanh"]),
        MultiDense([24, 8], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

#### 3.1.3. M: **50+50**
50% ReLU + 50% Sigmoid neuron activations per hidden layer.

In [ ]:
models["50+50"] = Sequential(
    [
        MultiDense([32, 32], activations=["relu", "tanh"]),
        MultiDense([32, 32], activations=["relu", "tanh"]),
        MultiDense([16, 16], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

#### 3.1.4. M: **25+75**
25% ReLU + 75% Sigmoid neuron activations per hidden layer.

In [ ]:
models["25+75"] = Sequential(
    [
        MultiDense([16, 48], activations=["relu", "tanh"]),
        MultiDense([16, 48], activations=["relu", "tanh"]),
        MultiDense([8, 24], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

#### 3.1.5. EM: **0+100**
0% ReLU + 100% Sigmoid neuron activations per hidden layer.

In [ ]:
models["0+100"] = Sequential(
    [
        MultiDense([0, 64], activations=["relu", "tanh"]),
        MultiDense([0, 64], activations=["relu", "tanh"]),
        MultiDense([0, 32], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

### 3.2. Models Compilation

In [ ]:
for tag, model in models.items():
    print(f"Compiling {tag} model...")

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"],
    )

    print(f"{tag} compiled!\n")


----

### 4.1. Load

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.boston_housing.load_data()

### 4.2. Preprocessing

In [ ]:
x_mean = x_train.mean(axis=0)
x_std = x_train.std(axis=0)

x_train = (x_train - x_mean) / x_std
x_test = (x_test - x_mean) / x_std

y_mean = y_train.mean()
y_std = y_train.std()

y_train_norm = (y_train - y_mean) / y_std
y_test_norm = (y_test - y_mean) / y_std


### 4.3. Training

In [ ]:
for tag, model in models.items():
    print(f"Training {tag} model...")

    model.fit(
        x_train,
        y_train_norm,
        epochs=200,
        batch_size=100,
        validation_split=0.1,
        verbose=0,
    )

    print(f"{tag} trained!\n")


### 4.2. Testing

In [ ]:
models_evaluation = {
    tag: model.evaluate(x_test, y_test_norm, verbose=0) for tag, model in models.items()
}

----

## 4. Experimentation

## 5. Conclusions

In [ ]:
for tag, metrics in models_evaluation.items():
    loss, mae = metrics
    mae_real = mae * y_std
    print(f"{tag} model -> Loss: {loss:.4f}, MAE: ${mae_real * 1000.0:.0f}")

----